In [3]:
import pandas as pd
import numpy as np
import joblib
import time
from rdkit import Chem
from rdkit.Chem import Descriptors, rdFingerprintGenerator

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

# ---------- 1. 합집합 구성 ----------
mp = pd.read_csv("./data/processed/crc_mp_c.csv").set_index("InChIKey")
bp = pd.read_csv("./data/processed/crc_bp_c.csv").set_index("InChIKey")
tox21 = pd.read_csv("./data/hazard/moleculenet_tox21.csv").drop_duplicates(subset="InChIKey", keep="first").set_index("InChIKey")
hazard_tasks = [c for c in tox21.columns if c != "SMILES"]

smiles_map = {}
for df in (mp, bp, tox21):
    for ikey, row in df["SMILES"].items():
        smiles_map.setdefault(ikey, row)

union_keys = list(smiles_map.keys())
compounds = pd.DataFrame({
    "InChIKey": union_keys,
    "SMILES": [smiles_map[k] for k in union_keys],
})
compounds["in_mp"] = compounds["InChIKey"].isin(mp.index)
compounds["in_bp"] = compounds["InChIKey"].isin(bp.index)
compounds["in_tox21"] = compounds["InChIKey"].isin(tox21.index)
print(f"합집합 크기: {len(compounds)}")

# ---------- 2. 모델/서술자 로드 ----------
mp_model = joblib.load("./models/mp_model.joblib")
mp_top40 = joblib.load("./models/top40_descriptors.joblib")
bp_model = joblib.load("./models/bp_model.joblib")
bp_top40 = joblib.load("./models/bp_top40_descriptors.joblib")
tox21_models = joblib.load("./models/tox21_models.joblib")
hazard_top40 = joblib.load("./models/hazard_top40_descriptors.joblib")

def slice_desc(all_d, names):
    d = np.array([all_d.get(n, 0.0) for n in names], dtype=float)
    d = np.nan_to_num(d, nan=0.0, posinf=0.0, neginf=0.0)
    return np.clip(d, -1e6, 1e6)

# ---------- 3. 컬럼별 측정값/예측값 채우기 ----------
t0 = time.time()
mp_vals, mp_src = [], []
bp_vals, bp_src = [], []
hazard_vals = {task: [] for task in hazard_tasks}
hazard_src = {task: [] for task in hazard_tasks}

for i, row in compounds.iterrows():
    smi, ikey = row["SMILES"], row["InChIKey"]
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        fp, all_d = None, None
    else:
        fp = np.array(morgan_gen.GetFingerprint(mol))
        all_d = Descriptors.CalcMolDescriptors(mol)

    # mp
    if row["in_mp"]:
        mp_vals.append(mp.loc[ikey, "Processed tmp/ºC"]); mp_src.append("measured")
    elif fp is not None:
        feat = np.concatenate([fp, slice_desc(all_d, mp_top40)]).reshape(1, -1)
        mp_vals.append(mp_model.predict(feat)[0]); mp_src.append("predicted")
    else:
        mp_vals.append(np.nan); mp_src.append("failed")

    # bp
    if row["in_bp"]:
        bp_vals.append(bp.loc[ikey, "Processed tbp/ºC"]); bp_src.append("measured")
    elif fp is not None:
        feat = np.concatenate([fp, slice_desc(all_d, bp_top40)]).reshape(1, -1)
        bp_vals.append(bp_model.predict(feat)[0]); bp_src.append("predicted")
    else:
        bp_vals.append(np.nan); bp_src.append("failed")

    # 유해성 12개 태스크
    hazard_feat = None
    for task in hazard_tasks:
        measured = tox21.loc[ikey, task] if row["in_tox21"] else np.nan
        if pd.notna(measured):
            hazard_vals[task].append(int(measured)); hazard_src[task].append("measured")
        elif fp is not None:
            if hazard_feat is None:
                hazard_feat = np.concatenate([fp, slice_desc(all_d, hazard_top40)]).reshape(1, -1)
            pred = tox21_models[task].predict_proba(hazard_feat)[0, 1]
            hazard_vals[task].append(pred); hazard_src[task].append("predicted")
        else:
            hazard_vals[task].append(np.nan); hazard_src[task].append("failed")

    if i % 2000 == 0:
        print(f"{i}/{len(compounds)} 진행, {time.time()-t0:.0f}초 경과")

compounds["mp"] = mp_vals
compounds["mp_source"] = mp_src
compounds["bp"] = bp_vals
compounds["bp_source"] = bp_src
for task in hazard_tasks:
    compounds[task] = hazard_vals[task]
    compounds[f"{task}_source"] = hazard_src[task]

compounds.to_csv("./data/processed/all_compounds_master.csv", index=False)
print(f"저장 완료: {len(compounds)}개, {compounds.shape[1]}개 컬럼, {time.time()-t0:.0f}초")

합집합 크기: 14008
0/14008 진행, 0초 경과
2000/14008 진행, 56초 경과
4000/14008 진행, 112초 경과
6000/14008 진행, 168초 경과
8000/14008 진행, 230초 경과


10000/14008 진행, 277초 경과
12000/14008 진행, 311초 경과
14000/14008 진행, 346초 경과
저장 완료: 14008개, 33개 컬럼, 346초
